# Storing axis metadata in xarray

```{seealso}
- [Zarr v3](zarr-v3.md) — `dimension_names` is a Zarr v3 feature used in this page
- [Coordinate Transformations](coordinate-transformations.md) — axes define the coordinate space; transforms operate on it
- [Axes → Dims](../mapping/axes-to-dims.md) — the mapping page for this spec topic
```

```{note}
This page covers how to store NGFF axis `type` and `unit` in xarray's data model.
It does not cover axis `name` → dim mapping (which is unambiguous) or coordinate
transforms (see [Coordinate Transformations](coordinate-transformations.md)).
For the full exploration that led to these conclusions, see
[Axes Experiments](axes-experiments.ipynb).
```

NGFF axes carry three fields per dimension: `name`, `type`, and `unit`.
The `name` → xarray dim mapping is unambiguous. But `type` (space, time, channel)
and `unit` (micrometer, millisecond) have no native home in xarray's data model.

## Prior art: CF Conventions and cf-xarray

This is not a new problem. The climate/forecast community solved it with
[CF Conventions](http://cfconventions.org/Data/cf-conventions/cf-conventions-1.11/cf-conventions.html),
which store axis identity and units as **attributes on coordinate variables**:

```python
# A typical CF-compliant coordinate
ds["lat"].attrs
# {'units': 'degrees_north', 'axis': 'Y', 'standard_name': 'latitude'}
```

[cf-xarray](https://cf-xarray.readthedocs.io/) builds a **stateless accessor**
on top of these attrs. The accessor re-reads attrs on every access (no cached
state) and provides a typed API:

```python
ds.cf["Y"]           # returns the Y-axis coordinate (whatever it's named)
ds.cf.axes           # {'X': ['lon'], 'Y': ['lat'], 'T': ['time']}
ds.cf["latitude"]    # looks up by standard_name
ds.cf.mean("X")      # rewrites to ds.mean(dim="lon") — argument translation
```

### How cf-xarray identifies axes

cf-xarray uses [a table of criteria](https://cf-xarray.readthedocs.io/en/latest/contributing.html#attribute-parsing)
(adapted from MetPy) that maps semantic keys to attribute-value pairs. For each
variable, it checks multiple attrs — `standard_name`, `axis`, `units`, and
several fallbacks — with any single match being sufficient (OR logic).

This heuristic is necessary because CF doesn't mandate a single canonical
attribute. NGFF is simpler: axis type is an **explicit field** (`"type": "space"`),
so no heuristic lookup is needed.

### Key cf-xarray design decisions

From studying the [cf-xarray source](https://github.com/xarray-contrib/cf-xarray):

- **Stateless accessor.** No cached state — every property re-scans attrs.
  Simple, but means every `ds.cf.axes` call iterates all variables.
- **Argument rewriting.** `ds.cf.mean("X")` translates to `ds.mean(dim="lon")`.
  This works by intercepting `__getattr__` and rewriting function arguments
  via a mapper table. Powerful pattern.
- **Multiple matches.** When multiple variables match a key (e.g., two variables
  with `axis="X"` on a staggered grid), `ds.cf["X"]` **raises an error** —
  you must use `ds.cf[["X"]]` to get a Dataset. This is documented as a pain point.
- **Fragile attrs.** Their top complaint: xarray operations can drop attrs.
  They recommend `xr.set_options(keep_attrs=True)` but it's global state.
- **No DataTree support.** The accessor is only registered on Dataset and DataArray.
- **Units are plain strings.** Stored as attrs, not interpreted. Actual unit-aware
  computation is left to [pint-xarray](https://pint-xarray.readthedocs.io/).
- **Custom criteria.** Users can register new vocabularies:
  ```python
  cf_xarray.set_options(custom_criteria={"ssh": {"name": "elev$"}})
  ds.cf["ssh"]  # finds variable matching regex "elev$"
  ```

### Where NGFF diverges from CF

| Aspect | CF | NGFF | Implication |
|--------|-----|------|-------------|
| Axis vocabulary | `X`, `Y`, `Z`, `T` — one per axis | `space`, `time`, `channel` — multiple `space` | cf-xarray assumes ≤1 var per axis letter; NGFF has 3 `space` axes routinely |
| Identification | inferred from multiple hints | **explicit** `type` field | NGFF doesn't need heuristics |
| Accessor lookup | `ds.cf["X"]` → single axis | `ds.ngff["space"]` → **3 axes** | need two-level lookup: by type then by name |
| DataTree | not supported | essential (multiscale) | must design for DataTree from day one |
| Scale/translation | no concept | `coordinateTransformations` | purely NGFF territory |

The multiple-space-axes issue is the key structural difference. In cf-xarray,
`ds.cf.mean("X")` means "average over the one X dimension." For NGFF,
`ds.ngff.mean("space")` would need to mean "average over x, y, and z" —
a one-to-many mapping that cf-xarray handles awkwardly (error on scalar key,
broadcast on method calls).

---

With this context, here are two concrete alternatives for storing NGFF axis
metadata in xarray. Both assume a **custom backend** handles round-tripping
to NGFF on disk — the question is purely about the **in-memory representation**.

See the [axes experiments page](axes-experiments.ipynb) for the full exploration
that led here.

In [1]:
import numpy as np
import xarray as xr

# Running example: 5D TCZYX
axes = [
    {"name": "t", "type": "time", "unit": "millisecond"},
    {"name": "c", "type": "channel"},
    {"name": "z", "type": "space", "unit": "micrometer"},
    {"name": "y", "type": "space", "unit": "micrometer"},
    {"name": "x", "type": "space", "unit": "micrometer"},
]
shape = (10, 3, 50, 256, 256)
dims = [a["name"] for a in axes]
data = np.zeros(shape, dtype=np.uint16)

## Recommendation, and the alternative we rejected

This page reaches a **recommendation**: store axis `type` and `unit` as
**CF-style attributes on each axis's coordinate variable** (Alternative A below),
surfaced through a typed `.ngff` accessor. The full rationale and the precise
mapping live in [Axes → Dims](../mapping/axes-to-dims.md); here we put the two
candidates side by side as executable evidence for *why* A wins.

Both alternatives answer the same question — **where should axis `type` and
`unit` live in xarray's data model?** — and both round-trip cleanly with a custom
backend, so the choice is purely about the in-memory representation. The strongest
runner-up was **Alternative B** (a separate `axis` dimension); it is kept here in
full because seeing it work is the clearest argument for the trade-off we made.

## Alternative A (recommended): CF-style coordinate attributes

```{admonition} This is the recommended approach
:class: tip
Axis `type` and `unit` are stored as attributes on each axis's **coordinate
variable** (`axis_type` and `units`) and read back through a typed `.ngff`
accessor. See [Axes → Dims](../mapping/axes-to-dims.md) for the full mapping.
```

**Motivation:** axis metadata follows the same storage pattern as CF
Conventions, so existing ecosystem tools (pint-xarray for units, cf-xarray's
accessor pattern) work out of the box and the convention is familiar to the
scientific Python community.

This follows the same storage model as CF and cf-xarray: axis type and unit
live as `.attrs` on each dimension coordinate. cf-xarray reads attrs like
`axis`, `standard_name`, and `units` — we store `axis_type` and `units`
in the same way.

```{note}
**Why `axis_type` instead of `type`?** We avoid `type` because it shadows
Python's builtin `type()` function, making it error-prone as a variable name
or dict key in user code. It also distinguishes NGFF's multi-valued vocabulary
(`"space"`, `"time"`, `"channel"`) from CF's single-letter `axis` attribute
(`"X"`, `"Y"`, `"Z"`, `"T"`).
```

We use `"axis_type"` rather than CF's single-letter `"axis"` because NGFF's
vocabulary (`"space"`, `"time"`, `"channel"`) doesn't map onto CF's
`X`/`Y`/`Z`/`T` — NGFF has three space axes that are all `"space"`, while
CF distinguishes `X`, `Y`, `Z`. We use `"units"` which is the standard CF key.

Just as cf-xarray builds a `.cf` accessor over CF attrs, we build an `.ngff`
accessor over these attrs — same architectural pattern, different vocabulary.
The accessor provides:

```python
ds.ngff.axes          # {'space': ['z', 'y', 'x'], 'time': ['t'], 'channel': ['c']}
ds.ngff["space"]      # returns all space coordinates (unlike cf-xarray, no error on multiple)
ds.ngff.mean("space") # rewrites to ds.mean(dim=["z", "y", "x"])
```

In [2]:
ds_cf = xr.Dataset({"image": (dims, data)})

for ax in axes:
    name = ax["name"]
    size = shape[dims.index(name)]
    coord_attrs = {}
    if "type" in ax:
        coord_attrs["axis_type"] = ax["type"]
    if "unit" in ax:
        coord_attrs["units"] = ax["unit"]
    ds_cf = ds_cf.assign_coords({name: (name, np.arange(size), coord_attrs)})

ds_cf

<xarray.Dataset> Size: 197MB
Dimensions:  (t: 10, c: 3, z: 50, y: 256, x: 256)
Coordinates:
  * t        (t) int64 80B 0 1 2 3 4 5 6 7 8 9
  * c        (c) int64 24B 0 1 2
  * z        (z) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49
  * y        (y) int64 2kB 0 1 2 3 4 5 6 7 8 ... 248 249 250 251 252 253 254 255
  * x        (x) int64 2kB 0 1 2 3 4 5 6 7 8 ... 248 249 250 251 252 253 254 255
Data variables:
    image    (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0

In [3]:
# Metadata lives on each coordinate
for name in dims:
    print(f"{name}: {dict(ds_cf[name].attrs)}")

t: {'axis_type': 'time', 'units': 'millisecond'}
c: {'axis_type': 'channel'}
z: {'axis_type': 'space', 'units': 'micrometer'}
y: {'axis_type': 'space', 'units': 'micrometer'}
x: {'axis_type': 'space', 'units': 'micrometer'}


In [4]:
# Querying
print("type of y:", ds_cf["y"].attrs["axis_type"])
print("unit of z:", ds_cf["z"].attrs["units"])
print()

# Find all space axes
space_axes = [d for d in ds_cf.sizes if ds_cf[d].attrs.get("axis_type") == "space"]
print("space axes:", space_axes)

type of y: space
unit of z: micrometer

space axes: ['z', 'y', 'x']


In [5]:
# Round-trip: reconstruct NGFF axes metadata
def axes_from_cf(ds: xr.Dataset) -> list[dict]:
    result = []
    for dim_name in ds.sizes:
        ax = {"name": dim_name}
        if dim_name in ds.coords:
            attrs = ds[dim_name].attrs
            if "axis_type" in attrs:
                ax["type"] = attrs["axis_type"]
            if "units" in attrs:
                ax["unit"] = attrs["units"]
        result.append(ax)
    return result

assert axes_from_cf(ds_cf) == axes
axes_from_cf(ds_cf)

[{'name': 't', 'type': 'time', 'unit': 'millisecond'},
 {'name': 'c', 'type': 'channel'},
 {'name': 'z', 'type': 'space', 'unit': 'micrometer'},
 {'name': 'y', 'type': 'space', 'unit': 'micrometer'},
 {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]

In [6]:
# Survives operations on data dimensions
sliced = ds_cf.isel(y=slice(0, 128))
print("after isel(y=...):", dict(sliced["y"].attrs))

selected = ds_cf.sel(t=5)
print("after sel(t=5): ", dict(selected["z"].attrs))

added = ds_cf + 1
print("after +1:       ", dict(added["y"].attrs))

after isel(y=...): {'axis_type': 'space', 'units': 'micrometer'}
after sel(t=5):  {'axis_type': 'space', 'units': 'micrometer'}
after +1:        {'axis_type': 'space', 'units': 'micrometer'}


### Summary

- Same storage model as CF/cf-xarray — proven pattern
- Metadata co-located with the coordinate it describes
- `units` key is standard CF; interoperates with pint-xarray for unit-aware computation
- **Not visible** in the Dataset repr — you have to inspect individual coordinate `.attrs` (the typed `.ngff` accessor restores discoverability)
- Coordinate attrs survive reductions, elementwise ops, slicing, `sel`, and `isel`, and round-trip losslessly through the Zarr v3 backend. The "attrs are fragile" folklore is **overstated**: the genuine loss vectors are narrow — binary ops between objects with *conflicting* attrs, `merge`/`concat` (keep-first default), and explicit `keep_attrs=False`. Storing metadata on the *coordinate* (not the data variable) avoids the worst of these — the same reason `rioxarray` keeps CRS on a coordinate.
- Requires dimension coordinates to exist (which we'll always have from coordinate transforms)
- A stateless `.ngff` accessor (like cf-xarray's `.cf`) sits on top

---

In [7]:
# Alternative A in a DataTree (the multiscale use case)
dt_cf = xr.DataTree.from_dict({
    "scale0": ds_cf,
    "scale1": ds_cf.isel(y=slice(0, 128), x=slice(0, 128)),
})
print(dt_cf)
print()
# Verify attrs propagate per-node
for node_name, node in dt_cf.children.items():
    print(f"{node_name} y attrs: {dict(node.dataset["y"].attrs)}")

<xarray.DataTree>
Group: /
├── Group: /scale0
│       Dimensions:  (t: 10, c: 3, z: 50, y: 256, x: 256)
│       Coordinates:
│         * t        (t) int64 80B 0 1 2 3 4 5 6 7 8 9
│         * c        (c) int64 24B 0 1 2
│         * z        (z) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49
│         * y        (y) int64 2kB 0 1 2 3 4 5 6 7 8 ... 248 249 250 251 252 253 254 255
│         * x        (x) int64 2kB 0 1 2 3 4 5 6 7 8 ... 248 249 250 251 252 253 254 255
│       Data variables:
│           image    (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
└── Group: /scale1
        Dimensions:  (t: 10, c: 3, z: 50, y: 128, x: 128)
        Coordinates:
          * t        (t) int64 80B 0 1 2 3 4 5 6 7 8 9
          * c        (c) int64 24B 0 1 2
          * z        (z) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49
          * y        (y) int64 1kB 0 1 2 3 4 5 6 7 8 ... 120 121 122 123 124 125 126 127
          * x        (x) int64 1

## Alternative B (rejected finalist): axis metadata on a separate dimension

**Motivation (as originally argued):** make axis metadata **visible in the repr**
and queryable with standard xarray operations, without inspecting `.attrs` on
individual coordinates — and sidestep the (as it turns out, overstated) "fragile
attrs" concern.

Instead of storing metadata in coordinate attrs, we introduce a new dimension
(`"axis"`) that indexes the axes themselves. Axis type and unit become regular
coordinates on this dimension — orthogonal to the data dimensions.

This is the strongest alternative to A, and the demo below shows it genuinely
works. We ultimately **rejected** it (see the summary and comparison that follow):
it is a novel pattern with no ecosystem precedent, the side table sits
disconnected from the dimensions it describes, and absent units hit the
`None → nan` object-array gotcha. Its headline draw — robustness against attrs
loss — is a weaker advantage once the fragility claim is put in proportion.

In [8]:
ds_sep = xr.Dataset({"image": (dims, data)})

# Dimension coordinates
for ax in axes:
    name = ax["name"]
    size = shape[dims.index(name)]
    ds_sep = ds_sep.assign_coords({name: np.arange(size)})

# Axis metadata on its own dimension
axis_names = [ax["name"] for ax in axes]
axis_types = [ax.get("type") for ax in axes]
axis_units = [ax.get("unit") for ax in axes]

ds_sep = ds_sep.assign_coords({
    "axis": axis_names,
    "axis_type": ("axis", axis_types),
    "axis_unit": ("axis", axis_units),
})

ds_sep

<xarray.Dataset> Size: 197MB
Dimensions:    (t: 10, c: 3, z: 50, y: 256, x: 256, axis: 5)
Coordinates:
  * t          (t) int64 80B 0 1 2 3 4 5 6 7 8 9
  * c          (c) int64 24B 0 1 2
  * z          (z) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49
  * y          (y) int64 2kB 0 1 2 3 4 5 6 7 ... 248 249 250 251 252 253 254 255
  * x          (x) int64 2kB 0 1 2 3 4 5 6 7 ... 248 249 250 251 252 253 254 255
  * axis       (axis) <U1 20B 't' 'c' 'z' 'y' 'x'
    axis_type  (axis) <U7 140B 'time' 'channel' 'space' 'space' 'space'
    axis_unit  (axis) object 40B 'millisecond' nan ... 'micrometer' 'micrometer'
Data variables:
    image      (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0

In [9]:
# Querying: look up by axis name
print("type of y:", ds_sep.axis_type.sel(axis="y").item())
print("unit of z:", ds_sep.axis_unit.sel(axis="z").item())
print()

# All space axes
space_mask = ds_sep.axis_type == "space"
space_axes = ds_sep.axis.where(space_mask, drop=True)
print("space axes:", space_axes.values)

type of y: space
unit of z: micrometer

space axes: ['z' 'y' 'x']


In [10]:
# Round-trip: reconstruct NGFF axes metadata
import pandas as pd

def axes_from_sep(ds: xr.Dataset) -> list[dict]:
    result = []
    for i, name in enumerate(ds.coords["axis"].values):
        ax = {"name": str(name)}
        axis_type = ds.coords["axis_type"].values[i]
        axis_unit = ds.coords["axis_unit"].values[i]
        if not pd.isna(axis_type):
            ax["type"] = str(axis_type)
        if not pd.isna(axis_unit):
            ax["unit"] = str(axis_unit)
        result.append(ax)
    return result

assert axes_from_sep(ds_sep) == axes
axes_from_sep(ds_sep)

[{'name': 't', 'type': 'time', 'unit': 'millisecond'},
 {'name': 'c', 'type': 'channel'},
 {'name': 'z', 'type': 'space', 'unit': 'micrometer'},
 {'name': 'y', 'type': 'space', 'unit': 'micrometer'},
 {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]

In [11]:
# Survives operations on data dimensions (axis dim is orthogonal)
sliced = ds_sep.isel(y=slice(0, 128))
print("after isel(y=...):")
print("  axis_type:", sliced.axis_type.values)
print("  axis_unit:", sliced.axis_unit.values)
print()

selected = ds_sep.sel(t=5)
print("after sel(t=5):")
print("  axis_type:", selected.axis_type.values)
print()

added = ds_sep + 1
print("after +1:")
print("  axis_type:", added.axis_type.values)

after isel(y=...):
  axis_type: ['time' 'channel' 'space' 'space' 'space']
  axis_unit: ['millisecond' nan 'micrometer' 'micrometer' 'micrometer']

after sel(t=5):
  axis_type: ['time' 'channel' 'space' 'space' 'space']

after +1:
  axis_type: ['time' 'channel' 'space' 'space' 'space']


In [12]:
# In a DataTree (the multiscale use case)
dt = xr.DataTree.from_dict({
    "scale0": ds_sep,
    "scale1": ds_sep.isel(y=slice(0, 128), x=slice(0, 128)),
})
print(dt)

<xarray.DataTree>
Group: /
├── Group: /scale0
│       Dimensions:    (t: 10, c: 3, z: 50, y: 256, x: 256, axis: 5)
│       Coordinates:
│         * t          (t) int64 80B 0 1 2 3 4 5 6 7 8 9
│         * c          (c) int64 24B 0 1 2
│         * z          (z) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49
│         * y          (y) int64 2kB 0 1 2 3 4 5 6 7 ... 248 249 250 251 252 253 254 255
│         * x          (x) int64 2kB 0 1 2 3 4 5 6 7 ... 248 249 250 251 252 253 254 255
│         * axis       (axis) <U1 20B 't' 'c' 'z' 'y' 'x'
│           axis_type  (axis) <U7 140B 'time' 'channel' 'space' 'space' 'space'
│           axis_unit  (axis) object 40B 'millisecond' nan ... 'micrometer' 'micrometer'
│       Data variables:
│           image      (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
└── Group: /scale1
        Dimensions:    (t: 10, c: 3, z: 50, y: 128, x: 128, axis: 5)
        Coordinates:
          * t          (t) int64 80B 0 1 2 3 4 5 

### Summary

- All axis metadata **visible in the repr** — immediately discoverable
- Queryable with standard xarray: `ds.axis_type.sel(axis="y")`
- Orthogonal to data dims — survives sel, isel, arithmetic
- Compact: one small dimension + 2 coordinates (not N scalar coords)
- Introduces a new dimension (`axis`) with no relationship to array data
- Missing values: `None` becomes `nan` in numpy object arrays — round-trip code must use `pd.isna()` rather than `is None`

**Why rejected:** the visibility and orthogonality are real, but they don't
outweigh adopting a novel, precedent-free pattern whose headline benefit
(avoiding attrs loss) shrinks once the fragility claim is corrected. Alternative A
gets coordinate-local metadata *and* free `cf-xarray`/`pint-xarray` interop from a
20-year-old convention.

---

## Comparison

| Property | A: CF-style coord attrs **(chosen)** | B: separate `axis` dim |
|----------|------------------------|------------------------|
| Visible in repr | no (hidden in `.attrs`) | **yes** |
| Queryable | `ds["y"].attrs["axis_type"]` | `ds.axis_type.sel(axis="y")` |
| Survives data operations | yes — reductions, elementwise, slicing, `sel`/`isel`; narrow loss vectors only (conflicting binary ops, `merge`/`concat`, `keep_attrs=False`) | yes (orthogonal dim) |
| Convention precedent | **CF Conventions** (proven, 20+ years) | novel |
| Accessor pattern | same as cf-xarray's `.cf` | different — data-based rather than attrs-based |
| pint-xarray interop | `units` attr is standard; pint-xarray can quantify directly | would need adapter |
| Namespace impact | none | adds `axis` dim + 2 coords |
| Missing values | key absent from attrs | `None` → `nan` in array |
| Rename sync | out of sync | out of sync |
| DataTree | works (attrs propagate per-node) | works (coords propagate per-node) |

Both approaches round-trip cleanly to NGFF axes metadata via a custom backend.
We choose **A**: coordinate-local metadata with free `cf-xarray`/`pint-xarray`
interop, at the cost of repr-visibility — which the typed `.ngff` accessor
restores for discoverability. See [Axes → Dims](../mapping/axes-to-dims.md) for
the full decision.

---

## Resolved decisions

The exploration above raised several questions; the book has since settled them.
Each is recorded here with the answer and where the full reasoning lives.

1. **Follow CF's storage model, or diverge?** → **Follow it.** Coordinate attrs
   are the chosen store (Alternative A); the "fragile attrs" objection is
   overstated (narrow loss vectors only), so the maturity and interop of the CF
   ecosystem win. See [Axes → Dims](../mapping/axes-to-dims.md).
2. **The multiple-`space`-axes problem** (cf-xarray errors on `ds.cf["X"]` when
   several variables match). → Handled by a **two-level lookup** in the typed
   accessor: select all axes of a type, or a single axis by name, so multiple
   `space` axes never collide the way `ds.cf["X"]` does.
3. **Is a new `"axis"` dimension acceptable?** → **No** — that was Alternative B,
   rejected as a precedent-free side table (see its summary above).
4. **Combine both** (CF attrs for `units`, a side dim for `axis_type`)? → **No.**
   Both fields live together on the coordinate; splitting them incurs the
   side-table downside for half the metadata with no offsetting gain.
5. **Accessor layer.** → **Yes** — a typed `.ngff` accessor is the public front
   door (read/build/select) and the single place that knows the storage
   convention. See [Axes → Dims](../mapping/axes-to-dims.md) for the layered API.
6. **DataTree.** → Both candidates propagate metadata per-node (shown above); the
   chosen design carries axis metadata on each node's coordinates. Multiscale
   specifics are in [Multiscales → DataTree](../mapping/multiscales-to-datatree.md).
7. **Interaction with coordinate transforms.** → Each axis's *coordinate* — the
   home for `type`/`unit` — is materialised from its `coordinateTransformations`;
   see [Transforms → Coords](../mapping/transforms-to-coords.md).

```{seealso}
- [Axes → Dims](../mapping/axes-to-dims.md) — the recommendation and rejected alternatives in full
- [Axes Experiments](axes-experiments.ipynb) — the eight-experiment exploratory record
```